# ALOHA Raw 查看与人工标注

本入口只读取 Raw，不需要 Prepared。只看轨迹执行第 1～4 步；需要人工标注时执行第 1～3、5～6 步，第 5 步自带播放器。

标注导出后，单独提交 `prepare_dataset.py` Ray Job 生成 Prepared 和 Samples；处理完成后打开 `aloha_prepared_viewer.ipynb` 查看结果。本 Notebook 不提交任务、不写 LakeSoul 表。

表中已有相机的图像（支持四路，兼容两路）和完整 episode 会读入 Notebook 内存。已有 PG/S3 环境配置即可；界面使用 ipywidgets，无需额外服务端口。缺少依赖时在单元格运行 `%pip install ipywidgets`。

相机按实际表结构自动识别，以两列排列；四路表显示四路，旧两路表仍可使用。这里只支持逐帧 JPEG 图片列，不直接解码 GOP 表。

新流程需将 LakeSoul ALOHA 示例的 `common.py` 放到 Notebook 当前工作目录，使用新导入的 Raw 表。导入元数据和标注版本自动存取 S3。

**表名对应关系：** 本 Notebook 读取 `aloha_velcro_raw_v1` 并发布标注，不生成 Prepared 表。随后按 README 运行 `prepare_dataset.py --raw-table aloha_velcro_raw_v1 --prepared-table aloha_velcro_prepared_v1`（其余必需参数见完整命令），再用 `aloha_prepared_viewer.ipynb` 查看 `aloha_velcro_prepared_v1`。修改示例表名时，各处对应参数必须同步修改。


## 第 1 步：配置 Raw 表和 Episode
修改配置后需重新执行本格和第 3 步，再重建播放器/标注界面。重读 Raw 会重建标注草稿，请先导出。


In [ ]:
import html
import os

import ipywidgets as widgets
from daft import col
from IPython.display import display
from lakesoul.catalog import LakeSoulCatalog
# 使用与 import_raw.py / prepare_dataset.py 同版的 common.py（放到当前工作目录）。
from common import read_import_manifest, publish_annotations, metadata_root

NAMESPACE = "default"
RAW_TABLE = "aloha_velcro_raw_v1"  # 与 import_raw.py 的 --raw-table、Prepare 的 --raw-table 一致
EPISODE_ID = "ep000000"


# 根据表中实际存在的相机列读取；兼容旧两路表和新四路表。
CAMERA_NAMES = ("cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist")


def make_camera_grid(width=360):
    images = {name: widgets.Image(format="jpeg", width=width) for name in CAMERA_NAMES}
    panels = {name: widgets.VBox([widgets.HTML(f"<b>{name}</b>"), images[name]],
                                layout=widgets.Layout(display="none"))
              for name in CAMERA_NAMES}
    grid = widgets.GridBox(list(panels.values()), layout=widgets.Layout(
        grid_template_columns=f"repeat(2, {width}px)", grid_gap="8px"))
    return images, panels, grid


def update_camera_images(images, panels, row=None):
    # 无此列则隐藏；清空或切换样本时同时清掉旧图片。
    row = {} if row is None else row
    for name, image in images.items():
        image.value = bytes(row.get(name) or b"")
        panels[name].layout.display = "" if name in row else "none"


## 第 2 步：连接 LakeSoul
读取已有 PG/S3 环境变量，不在 Notebook 中保存密钥。


In [ ]:
endpoint = os.getenv("AWS_ENDPOINT") or os.getenv("AWS_ENDPOINT_URL")

object_store_options = {
    "fs.s3a.path.style.access": "true",
}

if endpoint:
    object_store_options["fs.s3a.endpoint"] = endpoint

if os.getenv("AWS_ACCESS_KEY_ID"):
    object_store_options["fs.s3a.access.key"] = os.environ["AWS_ACCESS_KEY_ID"]

if os.getenv("AWS_SECRET_ACCESS_KEY"):
    object_store_options["fs.s3a.secret.key"] = os.environ["AWS_SECRET_ACCESS_KEY"]

catalog = LakeSoulCatalog.from_env(
    namespace=NAMESPACE,
    object_store_options=object_store_options,
)
print("Catalog 已创建，命名空间：", NAMESPACE)


## 第 3 步：读取 Raw 中的一条完整轨迹

按 `episode_id` 筛选，选出时间、状态、动作和表中已有的相机，再按原始 `frame_index` 排序。

`to_arrow().to_pylist()` 会真正执行查询，把这条轨迹的数据（包括 JPEG 图片字节）读到 Notebook 内存里。后面拖动滑块使用这些已加载记录，不会每滑动一次就查询数据库。长轨迹或大量高清图片会增加内存占用。

输出告诉你读到了多少行，以及首尾帧号。**首尾帧号不代表中间一定没有缺帧。**

In [ ]:
raw_rows = []
raw_table = catalog.table(RAW_TABLE)
raw_import_manifest = read_import_manifest(raw_table.path, NAMESPACE, RAW_TABLE)
camera_columns = [name for name in CAMERA_NAMES if name in raw_table.schema.names]
print("相机列：", camera_columns)
raw_rows = (
    raw_table
    .scan()
    .to_daft()
    .where(col("episode_id") == EPISODE_ID)
    .select(
        "episode_id",
        "frame_index",
        "timestamp",
        "observation_state",
        "action",
        *camera_columns,
    )
    .sort("frame_index")
    .to_arrow()
    .to_pylist()
)

if not raw_rows:
    raise ValueError(f"表 {RAW_TABLE!r} 中没有 Episode {EPISODE_ID!r}")

print("Raw 表：", RAW_TABLE)
print("已读取 Episode：", EPISODE_ID)
print("总帧数：", len(raw_rows))
print("帧范围：", raw_rows[0]["frame_index"], "-", raw_rows[-1]["frame_index"])

raw_context = (NAMESPACE, RAW_TABLE, EPISODE_ID)


## 第 4 步：浏览 Raw（可选）
展示表中已有相机的图像（支持四路，兼容两路）、原始帧号、时间戳和 state/action。滑块是记录位置，帧号另行显示；每 100 毫秒浏览一帧，不是原始 50 FPS。只为标注时可跳过本格，下一步自带播放器。


In [ ]:
# 重新运行这一格时，先停掉旧播放器。
if globals().get("play") is not None:
    play.playing = False
if globals().get("frame_link") is not None:
    frame_link.unlink()
if globals().get("viewer") is not None:
    viewer.close()

play = widgets.Play(value=0, min=0, max=0, step=1, interval=100, disabled=True)
frame_slider = widgets.IntSlider(
    value=0, min=0, max=0, step=1,
    description="帧位置：", continuous_update=False,
    disabled=True, layout=widgets.Layout(width="75%"),
)
frame_link = widgets.jslink((play, "value"), (frame_slider, "value"))
frame_info = widgets.HTML()
camera_images, camera_panels, camera_grid = make_camera_grid()
state_output = widgets.HTML()
action_output = widgets.HTML()
episode_rows = []


def format_vector(value):
    if value is None:
        return "null"
    return "[" + ", ".join(f"{float(item):.5f}" for item in value) + "]"


def render_frame(position):
    if not episode_rows:
        return
    row = episode_rows[position]
    timestamp = row.get("timestamp")
    timestamp_text = "null" if timestamp is None else f"{float(timestamp):.4f} s"
    notes = []
    if position > 0:
        previous = episode_rows[position - 1]
        if row["frame_index"] != previous["frame_index"] + 1:
            notes.append("帧号不连续：这里有缺口或重复帧")
        if row.get("valid_interval_id") != previous.get("valid_interval_id"):
            notes.append("进入另一个有效区间，训练窗口不能跨此边界")

    table_name = RAW_TABLE
    frame_info.value = (
        f"<b>表：</b>{html.escape(table_name)}　"
        f"<b>Episode：</b>{html.escape(str(row['episode_id']))}　"
        f"<b>frame_index：</b>{row['frame_index']}　"
        f"<b>timestamp：</b>{timestamp_text}　"
        f"<b>位置：</b>{position + 1}/{len(episode_rows)}<br>"
        f"<strong>{html.escape('；'.join(notes))}</strong>"
    )
    update_camera_images(camera_images, camera_panels, row)
    state_output.value = "<b>observation_state</b><pre>" + html.escape(format_vector(row.get("observation_state"))) + "</pre>"
    action_output.value = "<b>action</b><pre>" + html.escape(format_vector(row.get("action"))) + "</pre>"


def show_raw(change=None):
    global episode_rows
    play.playing = False
    play.disabled = frame_slider.disabled = True
    # 先清空数据，避免旧滑块位置触发越界回调。
    episode_rows = []
    play.value = frame_slider.value = 0
    episode_rows = raw_rows
    update_camera_images(camera_images, camera_panels)
    state_output.value = action_output.value = ""
    if not episode_rows:
        play.max = frame_slider.max = 0
        frame_info.value = "该表中没有此 Episode 的记录，请检查筛选结果或表名。"
        return
    play.max = frame_slider.max = len(episode_rows) - 1
    play.disabled = frame_slider.disabled = False
    render_frame(0)


frame_slider.observe(lambda change: render_frame(change["new"]), names="value")
viewer = widgets.VBox([
    widgets.HBox([play, frame_slider]),
    frame_info,
    camera_grid,
    state_output,
    action_output,
])
show_raw()
display(viewer)


## 第 5 步：分段草稿、逐段确认
本格把完整 Episode 划成连续且不重叠的区间。当前帧的行为显示在播放器上方；下方区间列表用于选择要审核的段。
本次 default.aloha_velcro_raw_20261009 的 ep000000 会预填人工粗标草稿：[0,218) approach、[218,467) manipulation、[467,700) release_or_retreat，全部待确认、默认保留。其他轨迹从整段 unknown 开始，这不是自动行为识别。

1. 点击区间列表，跳到该段起点；“播放所选段”只播放这段，“前一帧/后一帧”便于看边界。也可以用完整轨迹滑块看上下文。
2. 修改行为、说明或保留选项会立即更新草稿，并取消该段确认；不需要另外保存表单。
3. 修改开始或结束帧，会同步修改相邻段的共享边界，并取消两段的确认。第一段起点和最后一段终点固定。
4. “在当前帧切分”把当前画面所属区间分成两段；切分点是后一段第一帧。“与下一段合并”合并所选段，标签重置为 unknown，重新审核。
5. 看完点击“确认所选段”。全部确认后才能在第 6 步导出；unknown 也可以明确确认，表示无法判断行为。

行为确认与训练筛选分开：approach、manipulation、release_or_retreat 都可以保留。保留仍需通过 QC，行为边界本身不会切断连续保留的训练窗口。同一行为标签按相同策略保留/排除，导出时会检查。
同一数据对象重跑本格保留草稿；换轨迹或重新读取数据会重建草稿，因此请先导出。

In [ ]:
if raw_context != (NAMESPACE, RAW_TABLE, EPISODE_ID):
    raise ValueError("配置已改变，请先重新读取 Raw")
for name in ("play", "review_play", "annotation_play"):
    control = globals().get(name)
    if control is not None:
        control.playing = False
if globals().get("annotation_link") is not None:
    annotation_link.unlink()
for control in globals().get("annotation_widgets", []):
    control.close()

same_data = (globals().get("annotation_rows") is raw_rows
             and globals().get("annotation_context") == raw_context
             and globals().get("annotation_editor_version") == 2)
annotation_rows = raw_rows
annotation_context = raw_context
annotation_editor_version = 2
frame_ids = [row["frame_index"] for row in annotation_rows]
if not frame_ids or frame_ids != sorted(set(frame_ids)):
    raise ValueError("请先处理空轨迹、重复帧或帧号顺序问题")
frame_positions = {frame: i for i, frame in enumerate(frame_ids)}


def make_draft(start, end, behavior="unknown"):
    return dict(episode_id=annotation_context[2], start_frame=start,
                end_frame_exclusive=end, behavior=behavior, selected=True,
                reason="", annotation_source="manual_coarse", confirmed=False)


# ep000000 的边界不依赖表名或相机路数；预填草稿仍需人工确认。
use_velcro_draft = annotation_context[2] == "ep000000" and frame_ids == list(range(700))
# 重跑时升级尚未编辑的 unknown 草稿，保留已经修改或确认的标注。
untouched_draft = (same_data and use_velcro_draft
                   and annotation_segments == [make_draft(0, 700)])
if not same_data or untouched_draft:
    if use_velcro_draft:
        annotation_segments = [make_draft(s, e, label) for s, e, label in
            [(0,218,"approach"), (218,467,"manipulation"), (467,700,"release_or_retreat")]]
    else:
        annotation_segments = [make_draft(frame_ids[0], frame_ids[-1]+1)]

annotation_play = widgets.Play(min=0, max=len(annotation_rows)-1, interval=100)
annotation_slider = widgets.IntSlider(min=0, max=len(annotation_rows)-1,
    description="帧位置", continuous_update=False, layout=widgets.Layout(width="75%"))
annotation_link = widgets.jslink((annotation_play, "value"), (annotation_slider, "value"))
annotation_images, annotation_panels, annotation_grid = make_camera_grid()
annotation_info = widgets.HTML()
annotation_choice = widgets.Select(description="区间草稿", rows=6,
    layout=widgets.Layout(width="95%"))
annotation_start = widgets.IntText(description="开始帧")
annotation_end = widgets.IntText(description="结束(不含)")
annotation_behavior = widgets.Combobox(options=["approach", "grasp", "manipulation",
    "release_or_retreat", "preparation", "idle", "reset", "unknown"],
    description="行为", ensure_option=False, continuous_update=False)
annotation_selected = widgets.Checkbox(description="训练保留（仍需通过 QC）", value=True)
annotation_reason = widgets.Text(description="说明", continuous_update=False)
annotation_status = widgets.HTML()
annotation_summary = widgets.HTML()
confirm_button = widgets.Button(description="确认所选段", button_style="success")
split_button = widgets.Button(description="在当前帧切分")
merge_button = widgets.Button(description="与下一段合并")
segment_play_button = widgets.Button(description="播放所选段")
previous_frame_button = widgets.Button(description="前一帧")
next_frame_button = widgets.Button(description="后一帧")
editor_busy = False
segment_stop_position = None


def check_annotation_context():
    if annotation_context != (NAMESPACE, RAW_TABLE, EPISODE_ID) or annotation_rows is not raw_rows:
        raise ValueError("轨迹已改变，请重新运行读取和标注步骤")


def validate_annotation_partition(items, require_confirmed=False):
    check_annotation_context()
    expected = frame_ids[0]
    policies = {}
    for item in items:
        start, end = item["start_frame"], item["end_frame_exclusive"]
        if start != expected or start >= end:
            raise ValueError("区间必须连续、不重叠且非空")
        if start not in frame_positions or (end != frame_ids[-1]+1 and end not in frame_positions):
            raise ValueError("边界必须对应实际帧，末尾使用最后帧+1")
        label = item["behavior"].strip()
        if not label or "," in label:
            raise ValueError("行为标签不能为空或包含逗号")
        if label == "unknown" and not item["selected"]:
            raise ValueError("排除时请填写明确行为，避免影响其他未标注数据")
        if require_confirmed:
            if not item["confirmed"]:
                raise ValueError("还有待确认区间，请逐段检查后确认")
            if label in policies and policies[label] != item["selected"]:
                raise ValueError("同一标签不能同时保留和排除，请使用不同标签区分")
            policies[label] = item["selected"]
        expected = end
    if expected != frame_ids[-1]+1:
        raise ValueError("区间必须覆盖完整 Episode")


def render_annotation(change=None):
    global segment_stop_position
    position = annotation_slider.value
    row = annotation_rows[position]
    segment = next(a for a in annotation_segments
                   if a["start_frame"] <= row["frame_index"] < a["end_frame_exclusive"])
    state = "已确认" if segment["confirmed"] else "待确认草稿"
    annotation_info.value = (f"Episode: {html.escape(str(row['episode_id']))} | "
        f"frame_index: {row['frame_index']} | timestamp: {float(row['timestamp']):.3f}s<br>"
        f"<b>当前帧行为：{html.escape(segment['behavior'])}</b>（{state}），"
        f"区间 [{segment['start_frame']}, {segment['end_frame_exclusive']})")
    update_camera_images(annotation_images, annotation_panels, row)
    if segment_stop_position is not None and position >= segment_stop_position:
        annotation_play.playing = False
        segment_stop_position = None


def fill_editor():
    global editor_busy
    index = annotation_choice.value
    if index is None:
        return
    editor_busy = True
    try:
        item = annotation_segments[index]
        annotation_start.value = item["start_frame"]
        annotation_end.value = item["end_frame_exclusive"]
        annotation_start.disabled = index == 0
        annotation_end.disabled = index == len(annotation_segments)-1
        annotation_behavior.value = item["behavior"]
        annotation_selected.value = item["selected"]
        annotation_reason.value = item["reason"]
        merge_button.disabled = index == len(annotation_segments)-1
    finally:
        editor_busy = False


def refresh_annotation_list(index=0):
    global editor_busy
    editor_busy = True
    try:
        annotation_choice.options = [(f"[{a['start_frame']},{a['end_frame_exclusive']}) "
            f"{a['behavior']} | {'保留' if a['selected'] else '排除'} | "
            f"{'已确认' if a['confirmed'] else '待确认'}", i)
            for i, a in enumerate(annotation_segments)]
        annotation_choice.value = index
    finally:
        editor_busy = False
    fill_editor()
    count = sum(a["confirmed"] for a in annotation_segments)
    annotation_summary.value = f"已确认 {count}/{len(annotation_segments)} 段；全部确认后运行第 6 步导出。"
    render_annotation()


def choose_annotation(change):
    global segment_stop_position
    if editor_busy or change["new"] is None:
        return
    annotation_play.playing = False
    segment_stop_position = None
    fill_editor()
    annotation_slider.value = frame_positions[annotation_segments[change["new"]]["start_frame"]]
    render_annotation()


def change_annotation(change):
    if editor_busy or annotation_choice.value is None:
        return
    index = annotation_choice.value
    updated = [dict(a) for a in annotation_segments]
    try:
        check_annotation_context()
        item = updated[index]
        start, end = annotation_start.value, annotation_end.value
        if start != item["start_frame"]:
            if index == 0:
                raise ValueError("第一段起点固定")
            updated[index-1].update(end_frame_exclusive=start, confirmed=False)
        if end != item["end_frame_exclusive"]:
            if index == len(updated)-1:
                raise ValueError("最后一段终点固定")
            updated[index+1].update(start_frame=end, confirmed=False)
        item.update(start_frame=start, end_frame_exclusive=end,
            behavior=annotation_behavior.value.strip(), selected=annotation_selected.value,
            reason=annotation_reason.value, confirmed=False)
        validate_annotation_partition(updated)
        annotation_segments[:] = updated
        annotation_status.value = "草稿已更新；变动的区间需要重新确认。"
    except ValueError as error:
        annotation_status.value = html.escape(str(error))
    refresh_annotation_list(index)


def split_annotation(_):
    try:
        check_annotation_context()
        frame = annotation_rows[annotation_slider.value]["frame_index"]
        index = next(i for i,a in enumerate(annotation_segments)
                     if a["start_frame"] <= frame < a["end_frame_exclusive"])
        item = annotation_segments[index]
        if frame == item["start_frame"]:
            raise ValueError("当前帧已经是区间起点，请移到区间内部")
        left = dict(item, end_frame_exclusive=frame, confirmed=False)
        right = dict(item, start_frame=frame, confirmed=False)
        annotation_segments[index:index+1] = [left, right]
        refresh_annotation_list(index+1)
        annotation_status.value = "已切分，当前帧属于后一段；请分别检查标签并确认。"
    except ValueError as error:
        annotation_status.value = html.escape(str(error))


def merge_annotation(_):
    try:
        check_annotation_context()
        index = annotation_choice.value
        if index is None or index+1 >= len(annotation_segments):
            return
        first, second = annotation_segments[index:index+2]
        if first["selected"] != second["selected"]:
            raise ValueError("两段保留策略不同，请先明确统一策略再合并")
        merged = make_draft(first["start_frame"], second["end_frame_exclusive"])
        if not first["selected"]:
            raise ValueError("排除区间请保持分段，避免合并后丢失排除语义")
        annotation_segments[index:index+2] = [merged]
        refresh_annotation_list(index)
        annotation_status.value = "已合并，行为重置为 unknown，请重新标注并确认。"
    except ValueError as error:
        annotation_status.value = html.escape(str(error))


def confirm_annotation(_):
    try:
        validate_annotation_partition(annotation_segments)
        index = annotation_choice.value
        annotation_segments[index]["confirmed"] = True
        refresh_annotation_list(index)
        annotation_status.value = "已确认所选段，尚未写入文件；第 6 步统一导出。"
    except ValueError as error:
        annotation_status.value = html.escape(str(error))


def play_annotation(_):
    global segment_stop_position
    annotation_play.playing = False
    item = annotation_segments[annotation_choice.value]
    annotation_slider.value = frame_positions[item["start_frame"]]
    segment_stop_position = max(i for i,f in enumerate(frame_ids) if f < item["end_frame_exclusive"])
    annotation_play.repeat = False
    annotation_play.playing = True


def step_annotation(delta):
    global segment_stop_position
    annotation_play.playing = False
    segment_stop_position = None
    annotation_slider.value = max(0, min(len(annotation_rows)-1, annotation_slider.value+delta))


annotation_choice.observe(choose_annotation, names="value")
for control in (annotation_start, annotation_end, annotation_behavior, annotation_selected, annotation_reason):
    control.observe(change_annotation, names="value")
annotation_slider.observe(render_annotation, names="value")
confirm_button.on_click(confirm_annotation)
split_button.on_click(split_annotation)
merge_button.on_click(merge_annotation)
segment_play_button.on_click(play_annotation)
previous_frame_button.on_click(lambda _: step_annotation(-1))
next_frame_button.on_click(lambda _: step_annotation(1))
annotation_viewer = widgets.VBox([
    annotation_info, widgets.HBox([annotation_play, annotation_slider]),
    widgets.HBox([previous_frame_button, next_frame_button, segment_play_button]),
    annotation_grid,
    annotation_choice, widgets.HBox([annotation_start, annotation_end]),
    annotation_behavior, annotation_selected, annotation_reason,
    widgets.HBox([split_button, merge_button, confirm_button]),
    annotation_status, annotation_summary])
annotation_widgets = []
def remember_widgets(widget):
    annotation_widgets.append(widget)
    for child in getattr(widget, "children", ()):
        remember_widgets(child)
remember_widgets(annotation_viewer)
refresh_annotation_list()
display(annotation_viewer)

## 第 6 步：发布人工标注到 S3

全部区间确认后运行本格：标注和审核记录先上传为独立版本，再更新固定入口 `annotations/latest.json`。只替换当前 Episode 的标注，保留其他 Episode 已发布的记录。同一行为标签在所有 Episode 中必须使用一致的保留/排除策略。

无需下载标注再打包到 Ray。Prepare 根据 Raw 表自动读取这份标注和导入元数据；默认采用审核时的排除标签，也可用 `--exclude-behaviors` 显式覆盖。每次 Prepare 使用新的输出表名、路径。

请将 LakeSoul 示例目录中的最新 `common.py` 放到 Notebook 当前工作目录（只需随代码更新），并配置可访问 S3 的 AWS 环境变量。发布失败时不会提前更新入口。当前按单人依次审核设计，不支持多人同时发布的冲突合并。


In [ ]:
# 先验证当前画面对应的数据及所有区间的确认状态。
check_annotation_context()
validate_annotation_partition(annotation_segments, require_confirmed=True)
current_raw = catalog.table(RAW_TABLE)
current_manifest = read_import_manifest(current_raw.path, NAMESPACE, RAW_TABLE)
if current_manifest["import_id"] != raw_import_manifest["import_id"]:
    raise ValueError("Raw 已重新导入，请重新读取轨迹再标注")

# 先上传完整版本，再更新 latest.json；其余 Episode 的标注保留。
published = publish_annotations(
    current_raw.path, current_manifest, EPISODE_ID, annotation_segments,
)
print("发布成功，版本：", published["revision"])
print("标注：", published["annotations_uri"])
print("审核：", published["review_uri"])
print("固定入口：", metadata_root(current_raw.path) + "/annotations/latest.json")
print("默认排除标签：", published["excluded_behaviors"])
print("直接重新运行 prepare_dataset.py，不再传元数据或标注文件参数。")
